In [15]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 0, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 23, 59))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 182.09it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,1563618455000000301,NaN,NEWT,TRAD,2025-12-29 05:01:14+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
1,1563618456000000401,1.563618e+18,MODI,TRAD,2025-12-29 05:01:16+00:00,True,IR,None,I,True,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
2,1563618457000000501,1.563618e+18,MODI,TRAD,2025-12-29 05:01:19+00:00,False,IR,None,I,True,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
3,1563618458000000601,1.563618e+18,MODI,TRAD,2025-12-29 05:01:23+00:00,False,IR,None,I,True,...,,NaN,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
4,1563618910000000101,1.563616e+18,CORR,,2025-12-29 05:01:29+00:00,None,IR,None,I,True,...,,3.0,,,NaN,None,None,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11173,1575253883000000101,NaN,NEWT,TRAD,2025-12-30 04:56:51+00:00,None,IR,None,N,False,...,,NaN,,,NaN,None,None,QZ4HTNF4RVB5,NA/Swap Flt Flt AUD USD,AUD-BBSW vs USD-SOFR-OIS Compound
11174,1575167371000000101,NaN,NEWT,TRAD,2025-12-30 04:57:00+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZXF4PFZSLP7,NA/Swap OIS INR,INR-MIBOR-OIS Compound
11175,1575159075000000101,NaN,NEWT,TRAD,2025-12-30 04:57:10+00:00,None,IR,None,I,True,...,,NaN,,,NaN,None,None,QZV61TRS4HD9,NA/Swap OIS JPY,JPY-TONA-OIS-COMPOUND
11176,1575168481000000101,NaN,NEWT,TRAD,2025-12-30 04:57:34+00:00,None,IR,None,I,False,...,,NaN,,,NaN,None,None,QZ26BPG38C1K,NA/Swap Fxd Flt KRW,KRW-CD 91D


In [17]:
# from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
# USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path)

from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf.head(50)

Classifying Trades: 100%|██████████| 392/392 [00:00<00:00, 1783.79trade/s]


,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,upi_underlier_name,unique_product_identifier,platform_identifier,cleared,package_indicator,package_transaction_price,option_premium_amount,package_confidence,package_reason,package_legs_count
0,NEWT-TRAD,1568561409000000201,2025-12-29 09:17:55+00:00,2025-12-29,2026-12-29,SWAPTION_CHOOSER,USD-SOFR-OIS Compound 1D CONSTANT 1Y6Y CHOOSER...,1.000000e+07,USD,False,...,NA/Swap OIS USD,QZJBL64PCV75,XOFF,N,False,,"164,810",NaN,None,None
1,NEWT-NOVA,1568080997000000501,2025-12-29 09:19:10+00:00,2025-12-18,2026-08-21,SWAPTION_PAYER,USD-SOFR-OIS Compound 1Y CONSTANT 8M1Y PAYER E...,1.000000e+09,USD,False,...,NA/Swap OIS USD,QZZLNQ2D4JQT,BILT,N,False,,0,NaN,None,None
2,NEWT-TRAD,1570737966000000101,2025-12-29 13:05:00+00:00,2025-12-29,2026-02-27,SWAPTION_RECEIVER,USD-SOFR-COMPOUND 1D CONSTANT 2M10Y RECEIVER E...,2.000000e+08,USD,False,...,NA/Swap Fxd Flt USD,QZXZSN00ZVCG,BILT,N,False,,0,NaN,None,None
3,NEWT-TRAD,1570736565000000601,2025-12-29 13:05:00+00:00,2025-12-29,2026-02-27,SWAPTION_RECEIVER,USD-SOFR-COMPOUND 1D CONSTANT 2M10Y RECEIVER E...,2.000000e+08,USD,False,...,NA/Swap Fxd Flt USD,QZXZSN00ZVCG,BILT,N,False,,"3,610,000",NaN,None,None
4,NEWT-TRAD,1570700284000000301,2025-12-29 13:22:05+00:00,2025-12-29,2026-11-20,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 11M10Y RECEI...,5.000000e+07,USD,False,...,NA/Swap Fxd Flt USD,QZ7B7ZPS1LS5,BILT,N,False,,"2,447,500",NaN,None,None
5,NEWT-TRAD,1570700285000000401,2025-12-29 13:22:05+00:00,2025-12-29,2026-11-20,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 11M10Y RECEI...,5.000000e+07,USD,False,...,NA/Swap Fxd Flt USD,QZ7B7ZPS1LS5,BILT,N,False,,0,NaN,None,None
6,NEWT-TRAD,1570700287000000601,2025-12-29 13:22:06+00:00,2025-12-29,2026-08-19,SWAPTION_CHOOSER,USD-SOFR-OIS Compound 1D CONSTANT 8M10Y CHOOSE...,4.500000e+07,USD,False,...,NA/Swap OIS USD,QZMRJ6051HQB,BILT,N,False,,0,NaN,None,None
7,NEWT-TRAD,1570700286000000501,2025-12-29 13:22:06+00:00,2025-12-29,2026-08-19,SWAPTION_CHOOSER,USD-SOFR-OIS Compound 1D CONSTANT 8M10Y CHOOSE...,4.500000e+07,USD,False,...,NA/Swap OIS USD,QZMRJ6051HQB,BILT,N,False,,"1,831,500",NaN,None,None
8,TERM-ETRM,1570736561000000201,2025-12-29 13:36:16+00:00,2025-12-29,2026-11-20,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 11M10Y RECEI...,5.000000e+07,USD,False,...,NA/Swap Fxd Flt USD,QZ7B7ZPS1LS5,,N,False,,0,NaN,None,None
9,TERM-ETRM,1570736562000000301,2025-12-29 13:36:17+00:00,2025-12-29,2026-11-20,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 11M10Y RECEI...,5.000000e+07,USD,False,...,NA/Swap Fxd Flt USD,QZ7B7ZPS1LS5,,N,False,,0,NaN,None,None


In [18]:
sdf["package_type"].value_counts()

package_type
SWAPTION    143
STRADDLE    121
Name: count, dtype: int64

In [19]:
temp = sdf.copy()
temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
temp.to_excel("temp.xlsx", index=False)

In [43]:
temp = df[df["Dissemination Identifier"].isin(sdf["trade_id"])]
temp["Execution Timestamp"] = temp["Execution Timestamp"].astype(str)
temp["Event timestamp"] = temp["Event timestamp"].astype(str)
temp.to_excel("swaption_trades.xlsx",index=False)

C:\Users\chris\AppData\Local\Temp\ipykernel_79052\2697270985.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\chris\AppData\Local\Temp\ipykernel_79052\2697270985.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
https://github.com/yieldcurvemonkey/SwapPulse/tree/3a2fdbf844e75d048f802b6f099ca8707e5879be

In [10]:
df[df["Dissemination Identifier"] == 1570700287000000601].iloc[-1].to_dict()

{'Dissemination Identifier': 1570700287000000601,
 'Original Dissemination Identifier': nan,
 'Action type': 'NEWT',
 'Event type': 'TRAD',
 'Event timestamp': Timestamp('2025-12-29 13:22:06+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'N',
 'Mandatory clearing indicator': False,
 'Execution Timestamp': Timestamp('2025-12-29 13:11:18+0000', tz='UTC'),
 'Effective Date': Timestamp('2025-12-29 00:00:00'),
 'Expiration Date': Timestamp('2026-08-19 00:00:00'),
 'Maturity date of the underlier': datetime.date(2036, 8, 21),
 'Non-standardized term indicator': False,
 'Platform identifier': 'BILT',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': False,
 'Large notional off-facility swap election indicator': False,
 'Notional amount-Leg 1': '45,000,000',
 'Notional amount-Leg 2': '45,000,000',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': 'USD',
 'Notional quantity-Leg 1': None,
 'N